# ELEC0447: Newton-Raphson Method in 1D
**Interactive Demonstration and Geometric Progression**

This notebook accompanies the lecture slides on *Introduction to Power Flow Analysis*.
It illustrates how Newton-Raphson finds roots through successive tangent approximations and demonstrates its quadratic convergence rate.

## 1. Problem Formulation

We want to find $x$ such that the mismatch function is zero:
$$g(x) = c - f(x) = 0$$

In our lecture example, $c = 4$ and $f(x) = x^3$, so:
$$g(x) = 4 - x^3 = 0 \implies x^* = \sqrt[3]{4} \approx 1.587401$$

### First-Order Taylor Series Linearization
Around the current iterate $x^{(i)}$:
$$g(x^{(i)} + \Delta x) \approx g(x^{(i)}) + g'(x^{(i)}) \Delta x = 0$$

Since $g'(x) = -f'(x) = -3x^2$, the Newton update step is:
$$\Delta x^{(i)} = -\frac{g(x^{(i)})}{g'(x^{(i)})} = \frac{c - f(x^{(i)})}{f'(x^{(i)})} = \frac{4 - (x^{(i)})^3}{3 (x^{(i)})^2}$$
$$x^{(i+1)} = x^{(i)} + \Delta x^{(i)}$$

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

try:
    import ipywidgets as widgets
    from ipywidgets import interact
    has_widgets = True
except ImportError:
    has_widgets = False
    print('ipywidgets not installed; running in standard plotting mode.')

In [ ]:
# Define function and derivative
c = 4.0
def f(x):
    return x**3

def df(x):
    return 3.0 * x**2

def g(x):
    return c - f(x)

def dg(x):
    return -df(x)

exact_root = 4.0**(1.0/3.0)
print(f'Exact root: x* = {exact_root:.6f}')

## 2. Step-by-Step Newton-Raphson Solver

In [ ]:
def run_newton(x0, max_iter=6, tol=1e-6):
    records = []
    x = float(x0)
    for i in range(max_iter):
        gx = g(x)
        dgx = dg(x)
        err = abs(x - exact_root)
        if abs(dgx) < 1e-12:
            records.append({'Iter': i, 'x': x, 'g(x)': gx, "g'(x)": dgx, 'dx': 0.0, '|x - x*|': err})
            break
        dx = -gx / dgx  # = (c - f(x)) / df(x)
        records.append({'Iter': i, 'x': x, 'g(x)': gx, "g'(x)": dgx, 'dx': dx, '|x - x*|': err})
        if abs(gx) <= tol:
            break
        x = x + dx
    return pd.DataFrame(records)

## 3. Interactive Visualization of Tangents and Convergence
Adjust the initial guess $x^{(0)}$ and observe the tangent intersections.

In [ ]:
def plot_newton(x0=3.5, num_steps=3):
    df_iters = run_newton(x0, max_iter=num_steps+1)
    
    x_vals = np.linspace(0.8, max(4.0, x0 + 0.5), 300)
    y_vals = g(x_vals)
    
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))
    
    # Subplot 1: Geometric Tangents
    ax1.plot(x_vals, y_vals, label='$g(x) = 4 - x^3$', color='#1a73e8', lw=2.5)
    ax1.axhline(0, color='black', lw=1, ls='--', alpha=0.7)
    ax1.scatter([exact_root], [0], color='#10b981', s=80, zorder=5, label=f'Root $x^* \approx {exact_root:.3f}$')
    
    colors = ['#ea4335', '#f59e0b', '#8b5cf6', '#06b6d4', '#ec4899']
    for k in range(min(num_steps, len(df_iters) - 1)):
        xk = df_iters.loc[k, 'x']
        gk = df_iters.loc[k, 'g(x)']
        x_next = xk + df_iters.loc[k, 'dx']
        col = colors[k % len(colors)]
        
        # Vertical line to curve
        ax1.plot([xk, xk], [0, gk], color=col, ls=':', lw=1.5)
        ax1.scatter([xk], [gk], color=col, s=40, zorder=4)
        
        # Tangent line to x-axis
        ax1.plot([xk, x_next], [gk, 0], color=col, lw=2, label=f'Step {k}: $x^{{({k})}}={xk:.3f} \to x^{{({k+1})}}={x_next:.3f}$')
        ax1.scatter([x_next], [0], color=col, s=40, zorder=4)
    
    ax1.set_xlabel('x', fontsize=11)
    ax1.set_ylabel('g(x)', fontsize=11)
    ax1.set_title('Geometric Newton-Raphson Tangent Steps', fontsize=12, fontweight='bold')
    ax1.grid(True, alpha=0.3)
    ax1.legend(loc='lower left', fontsize=9)
    
    # Subplot 2: Quadratic Convergence Log Error
    errors = df_iters['|x - x*|'].values
    ax2.semilogy(df_iters['Iter'], errors, marker='o', color='#00843B', lw=2)
    ax2.set_xlabel('Iteration $i$', fontsize=11)
    ax2.set_ylabel('Error $|x^{(i)} - x^*|$', fontsize=11)
    ax2.set_title('Quadratic Convergence Rate', fontsize=12, fontweight='bold')
    ax2.grid(True, alpha=0.3, which='both')
    
    plt.tight_layout()
    plt.show()
    
    # Display iteration table
    display(df_iters.style.format({'x': '{:.6f}', 'g(x)': '{:.6f}', "g'(x)": '{:.4f}', 'dx': '{:.6f}', '|x - x*|': '{:.2e}'}))

if has_widgets:
    interact(plot_newton, x0=widgets.FloatSlider(value=3.5, min=1.0, max=4.5, step=0.1, description='x₀:'),
                          num_steps=widgets.IntSlider(value=3, min=1, max=5, step=1, description='Steps:'))
else:
    plot_newton(x0=3.5, num_steps=3)

## 4. Bridge to Power Systems: 2-Bus Power Flow Mismatch

Consider active power transmission between two buses with voltages $V_1=1.0\text{ pu}, V_2=1.0\text{ pu}$ over a line of reactance $X=0.2\text{ pu}$:
$$P(\theta) = \frac{V_1 V_2}{X} \sin(\theta) = 5.0 \sin(\theta)$$

We want to find the angle difference $\theta$ that transfers specified power $P_0 = 2.5\text{ pu}$:
$$g(\theta) = P_0 - P(\theta) = 2.5 - 5.0 \sin(\theta) = 0 \implies \theta^* = \arcsin(0.5) = \frac{\pi}{6} \approx 0.5236\text{ rad} \ (30^\circ)$$

In [ ]:
P_spec = 2.5
def P_transfer(th):
    return 5.0 * np.sin(th)

def dP_dth(th):
    return 5.0 * np.cos(th)

th_exact = np.pi / 6.0

# Solve with Newton-Raphson starting from flat start (th = 0)
th = 0.0
print('Flat start: theta^(0) = 0.0 rad (0.0 deg)')
for i in range(5):
    mismatch = P_spec - P_transfer(th)
    J = -dP_dth(th)
    dth = -mismatch / J  # = (P_spec - P(th)) / (dP/dth)
    print(f'Iter {i}: theta = {th:.6f} rad ({np.degrees(th):.3f} deg), Mismatch = {mismatch:.6e} pu, dtheta = {dth:.6f}')
    th += dth
    if abs(mismatch) < 1e-6:
        print(f'Converged in {i+1} iterations! Final theta = {np.degrees(th):.3f} deg')
        break